# Quick MEM — 1D Mechanical Earth Model Workflow

End-to-end demo of **GeomechPy** following the standard MEM sequence:

1. Data input (pandas)
2. Overburden stress
3. Lithology (toolbox)
4. Pore pressure
5. Dynamic elastic properties
6. Static elastic properties
7. Rock strength
8. Horizontal stresses + stress-tensor rotation
9. Wellbore stability (breakout / breakdown)

Synthetic log data is generated so the notebook runs without external files.


## 0. Setup & imports


In [ ]:
import sys
from pathlib import Path

# repo root = two levels up from this notebook (example/Project/)
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "Project":
    REPO_ROOT = REPO_ROOT.parent.parent
elif REPO_ROOT.name == "example":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "example"))  # lithology in example/toolbox.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from geomechpy.overburden_stress import OverburdenStressCalculation
from geomechpy.pore_pressure import PorePressureCalculation
from geomechpy.elastic_properties import ElasticPropertiesConverter
from geomechpy.static_elastic_properties import StaticElasticPropertiesConverter
from geomechpy.rock_strength import RockStrengthPropertiesConverter
from geomechpy.stress_calculations import HorizontalStressesCalculation
from geomechpy.wellbore_stability import WellboreStabilityCalculation
from geomechpy.toolbox import rotate_stress_to_shmax, rotate_nev_to_toh
from toolbox import determine_lithology_array  # example/toolbox.py

print("Imports OK — repo root:", REPO_ROOT)


## 1. Data input

Synthetic onshore well log (TVD from 5 000 – 8 000 ft).
Columns mimic typical LAS curves used in a 1-D MEM.


In [ ]:
np.random.seed(42)
n = 61
tvd = np.linspace(5000.0, 8000.0, n)  # ft

# Synthetic curves with mild depth trends + noise
gr = 40 + 40 * np.sin(np.linspace(0, 4 * np.pi, n)) + np.random.normal(0, 5, n)  # gAPI
gr = np.clip(gr, 15, 140)

rhob = 2.35 + 0.00004 * (tvd - 5000) + np.random.normal(0, 0.02, n)  # g/cm3
dtco = 90 - 0.005 * (tvd - 5000) + np.random.normal(0, 2, n)       # us/ft
dtsh = 160 - 0.008 * (tvd - 5000) + np.random.normal(0, 3, n)      # us/ft
nphi = 0.18 - 0.00002 * (tvd - 5000) + np.random.normal(0, 0.01, n) # fraction
nphi = np.clip(nphi, 0.05, 0.30)

# Optional coal / limestone flags (mostly False)
coal_flag = [False] * n
limestone_flag = [False] * n
coal_flag[20] = True
limestone_flag[45] = True

df = pd.DataFrame({
    "TVD": tvd,
    "GR": gr,
    "RHOB": rhob,
    "DTCO": dtco,
    "DTSH": dtsh,
    "NPHI": nphi,
    "COAL": coal_flag,
    "LIME": limestone_flag,
})
df.head()


## 2. Overburden stress (onshore)


In [ ]:
AIR_GAP = 30.0  # ft KB to ground
LITHOSTATIC_GRAD = 1.05  # psi/ft

df["SV"] = OverburdenStressCalculation.calculate_overburden_stress_onshore_array(
    tvd=df["TVD"].tolist(),
    lithostatic_gradient=LITHOSTATIC_GRAD,
    air_gap=AIR_GAP,
)
df[["TVD", "SV"]].head()


## 3. Lithology (mechanical stratigraphy from GR)


In [ ]:
df["LITH"] = determine_lithology_array(
    gamma_ray=df["GR"].tolist(),
    gr_threshold=75.0,
    coal_flag=df["COAL"].tolist(),
    limestone_flag=df["LIME"].tolist(),
)
# 0=Sand, 1=Shale, 2=Limestone, 6=Coal
df["LITH_NAME"] = df["LITH"].map({0: "Sand", 1: "Shale", 2: "Limestone", 6: "Coal"})
df[["TVD", "GR", "LITH", "LITH_NAME"]].value_counts("LITH_NAME")


## 4. Pore pressure (onshore hydrostatic)


In [ ]:
PP_GRAD = 0.465  # psi/ft

df["PP"] = PorePressureCalculation.calculate_pore_pressure_onshore_array(
    tvd=df["TVD"].tolist(),
    formation_pore_pressure_gradient=PP_GRAD,
    air_gap=AIR_GAP,
)
df[["TVD", "PP"]].head()


## 5. Dynamic elastic properties

From compressional / shear slowness + bulk density.


In [ ]:
# density must be kg/m3 for the converter
density_kgm3 = (df["RHOB"] * 1000).tolist()

dyn_props = ElasticPropertiesConverter.convert_dynamic_elastic_properties_from_slowness_array(
    p_wave_slowness=df["DTCO"].tolist(),
    s_wave_slowness=df["DTSH"].tolist(),
    density=density_kgm3,
)

df["YME_DYN_Pa"] = [p.youngs_modulus for p in dyn_props]
df["PR_DYN"] = [p.poissons_ratio for p in dyn_props]
df["G_DYN_Pa"] = [p.shear_modulus for p in dyn_props]

# Convert Pa to Mpsi for static correlations that expect Mpsi
PA_TO_MPSI = 1.0 / 6.894757e9
df["YME_DYN_Mpsi"] = df["YME_DYN_Pa"] * PA_TO_MPSI
df[["TVD", "YME_DYN_Mpsi", "PR_DYN"]].head()


## 6. Static elastic properties

Bradford correlation (dynamic to static YME) + simple PR scaling.


In [ ]:
df["YME_STA_Mpsi"] = StaticElasticPropertiesConverter.dyn2sta_yme_bradord_array(
    yme_dyn=df["YME_DYN_Mpsi"].tolist()
)
df["PR_STA"] = StaticElasticPropertiesConverter.dyn2sta_poissons_ratio_array(
    pr_dyn=df["PR_DYN"].tolist(),
    multiplier=1.0,
)
df["BIOT"] = [
    StaticElasticPropertiesConverter.biot_coefficient_constant_law(1.0)
    for _ in range(len(df))
]
df[["TVD", "YME_STA_Mpsi", "PR_STA", "BIOT"]].head()


## 7. Rock strength (UCS, tensile strength, friction angle)


In [ ]:
df["UCS"] = RockStrengthPropertiesConverter.convert_yme_sta_to_ucs_plumb_array(
    yme_sta=df["YME_STA_Mpsi"].tolist()
)
df["TSTR"] = RockStrengthPropertiesConverter.convert_ucs_to_tstr_array(
    ucs=df["UCS"].tolist(),
    multiplier=0.15,
)
df["FANG"] = RockStrengthPropertiesConverter.convert_friction_angle_lal_array(
    dtco=df["DTCO"].tolist()
)
df[["TVD", "UCS", "TSTR", "FANG"]].head()


## 8. Horizontal stresses (poroelastic) + stress-tensor rotation

Poroelastic model gives Shmin / Shmax, then rotate principal stresses into NEV frame.


In [ ]:
hs_list = HorizontalStressesCalculation.calculate_poroelastic_horizontal_stresses_array(
    overburden_stress=df["SV"].tolist(),
    pore_pressure=df["PP"].tolist(),
    poisson_ratio=df["PR_STA"].tolist(),
    youngs_modulus=df["YME_STA_Mpsi"].tolist(),
    biot_coefficient=df["BIOT"].tolist(),
    EX=0.0001,
    EY=0.0005,
)
df["SHMIN"] = [h.shmin for h in hs_list]
df["SHMAX"] = [h.shmax for h in hs_list]
df["Q_FACTOR"] = [h.q_factor for h in hs_list]
df["SHMAX_SHMIN_RATIO"] = [h.shmax_shmin_ratio for h in hs_list]

# Example rotation at mid-depth
mid = len(df) // 2
SHMAX_AZIMUTH = 45.0  # deg from North
stress_nev = rotate_stress_to_shmax(
    shmin=df["SHMIN"].iloc[mid],
    shmax=df["SHMAX"].iloc[mid],
    svert=df["SV"].iloc[mid],
    shmax_azimuth=SHMAX_AZIMUTH,
)
print("Stress tensor NEV at mid-depth (psi):")
print(np.round(stress_nev, 1))
df[["TVD", "SHMIN", "SHMAX", "Q_FACTOR"]].head()


## 9. Wellbore stability (vertical well)

Breakout (Mohr-Coulomb) and breakdown (tensile) pressures.


In [ ]:
df["PW_BREAKOUT"] = (
    WellboreStabilityCalculation
    .calculate_breakout_calculation_vertical_well_mohr_coulomb_analytical_array(
        shmax=df["SHMAX"].tolist(),
        shmin=df["SHMIN"].tolist(),
        pprs=df["PP"].tolist(),
        overburden_stress=df["SV"].tolist(),
        ucs=df["UCS"].tolist(),
        fang=df["FANG"].tolist(),
        pr_sta=df["PR_STA"].tolist(),
    )
)
df["PW_BREAKDOWN"] = (
    WellboreStabilityCalculation
    .calculate_breakdown_calculation_vertical_well_analytical_array(
        shmax=df["SHMAX"].tolist(),
        shmin=df["SHMIN"].tolist(),
        pprs=df["PP"].tolist(),
        tstr=df["TSTR"].tolist(),
    )
)
df[["TVD", "PW_BREAKOUT", "PW_BREAKDOWN", "PP"]].head()


## Summary plot — key MEM curves


In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(14, 8), sharey=True)

axes[0].plot(df["GR"], df["TVD"], "g-")
axes[0].set_xlabel("GR (gAPI)")
axes[0].invert_yaxis()
axes[0].set_ylabel("TVD (ft)")

axes[1].plot(df["SV"] / 1000, df["TVD"], "k-", label="Sv")
axes[1].plot(df["SHMAX"] / 1000, df["TVD"], "r-", label="SHmax")
axes[1].plot(df["SHMIN"] / 1000, df["TVD"], "b-", label="Shmin")
axes[1].plot(df["PP"] / 1000, df["TVD"], "c-", label="Pp")
axes[1].set_xlabel("Stress (ksi)")
axes[1].legend(fontsize=7)

axes[2].plot(df["YME_STA_Mpsi"], df["TVD"], "m-")
axes[2].set_xlabel("YME_sta (Mpsi)")

axes[3].plot(df["PR_STA"], df["TVD"], "orange")
axes[3].set_xlabel("PR_sta")

axes[4].plot(df["UCS"] / 1000, df["TVD"], "brown")
axes[4].set_xlabel("UCS (ksi)")

axes[5].plot(df["PW_BREAKOUT"] / 1000, df["TVD"], "r-", label="Breakout")
axes[5].plot(df["PW_BREAKDOWN"] / 1000, df["TVD"], "b-", label="Breakdown")
axes[5].plot(df["PP"] / 1000, df["TVD"], "c--", label="Pp")
axes[5].set_xlabel("Mud window (ksi)")
axes[5].legend(fontsize=7)

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.suptitle("Quick MEM — Synthetic Well", fontsize=12)
plt.tight_layout()
plt.show()


## Done

The DataFrame `df` now holds a complete 1-D MEM at every depth sample.
Export with:

```python
df.to_csv("quick_mem_results.csv", index=False)
```
